In [1]:
import eurostat
import pandas as pd
import numpy as np
import requests, json

%load_ext autoreload
%autoreload 2

DEMOGRAPH = "demo_r_d2jan"
ENVIRONMENT = "env_air_gge"
DBS = [DEMOGRAPH, ENVIRONMENT]

In [ ]:
data = eurostat.get_data_df(DEMOGRAPH)
data.to_csv("demo.csv", index=False)
data = eurostat.get_data_df(ENVIRONMENT)
data.to_csv("env.csv", index=False)

# Dimensions and label meanings

In [24]:
for DB in DBS:
    print(DB)

    dims = eurostat.get_pars(DB)
    print(f"Dimension labels: {dims}\n")

    for dim in dims:
        print(f"{dim}'s label meanings:\n{eurostat.get_dic(DB, dim)}\n")


demo_r_d2jan
Dimension labels: ['freq', 'unit', 'sex', 'age', 'geo']

freq's label meanings:
[('P', 'Pluri-annual'), ('A', 'Annual'), ('S', 'Half-yearly, semesterly'), ('Q', 'Quarterly'), ('M', 'Monthly'), ('W', 'Weekly'), ('B', 'Daily - business week'), ('D', 'Daily'), ('H', 'Hourly'), ('I', 'Irregular / A-periodic'), ('NAP', 'Not applicable'), ('A3', 'Triannual')]

unit's label meanings:
[('TOTAL', 'Total'), ('NR', 'Number'), ('NR_HAB', 'Number per inhabitant'), ('NR_HHAB', 'Number per hundred inhabitants'), ('NR_HTHAB', 'Number per hundred thousand persons'), ('THS', 'Thousand'), ('MIO', 'Million'), ('BN', 'Billion'), ('CT', 'Euro cent'), ('EUR', 'Euro'), ('THS_EUR', 'Thousand euro'), ('MIO_EUR', 'Million euro'), ('BN_EUR', 'Billion euro'), ('MEUR_KP', 'Million euro at constant prices'), ('MEUR_KP21', 'Million euro (at constant 2021 prices)'), ('MEUR_KP15', 'Million euro (at constant 2015 prices)'), ('MEUR_KP10', 'Million euro (at constant 2010 prices)'), ('MEUR_KP00', 'Million euro

In [4]:
df = eurostat.get_data_df(DEMOGRAPH)
#print(df.columns)

geo_data_demo = sorted(df["geo\\TIME_PERIOD"].unique())
geo_demo_nuts2 = sorted({g for g in geo_data_demo if len(g) == 4 and not g.endswith("ZZ") and not g.endswith("XX") and g not in {"EU28", "EFTA"}})
print(f"Demographic's NUTS2 region labels: {geo_demo_nuts2}")

df_env = eurostat.get_data_df(ENVIRONMENT)
#print(df.columns)

geo_data_env = sorted(df_env["geo\\TIME_PERIOD"].unique())
geo_env_country = sorted(set([g for g in geo_data_env if len(g) == 2]))
print(f"Environmental emission country labels: {geo_env_country}")

Demographic's NUTS2 region labels: ['AL01', 'AL02', 'AL03', 'AT11', 'AT12', 'AT13', 'AT21', 'AT22', 'AT31', 'AT32', 'AT33', 'AT34', 'BE10', 'BE21', 'BE22', 'BE23', 'BE24', 'BE25', 'BE31', 'BE32', 'BE33', 'BE34', 'BE35', 'BG31', 'BG32', 'BG33', 'BG34', 'BG41', 'BG42', 'CH01', 'CH02', 'CH03', 'CH04', 'CH05', 'CH06', 'CH07', 'CY00', 'CZ01', 'CZ02', 'CZ03', 'CZ04', 'CZ05', 'CZ06', 'CZ07', 'CZ08', 'DE11', 'DE12', 'DE13', 'DE14', 'DE21', 'DE22', 'DE23', 'DE24', 'DE25', 'DE26', 'DE27', 'DE30', 'DE40', 'DE50', 'DE60', 'DE71', 'DE72', 'DE73', 'DE80', 'DE91', 'DE92', 'DE93', 'DE94', 'DEA1', 'DEA2', 'DEA3', 'DEA4', 'DEA5', 'DEB1', 'DEB2', 'DEB3', 'DEC0', 'DED2', 'DED4', 'DED5', 'DEE0', 'DEF0', 'DEG0', 'DK01', 'DK02', 'DK03', 'DK04', 'DK05', 'EE00', 'EL30', 'EL41', 'EL42', 'EL43', 'EL51', 'EL52', 'EL53', 'EL54', 'EL61', 'EL62', 'EL63', 'EL64', 'EL65', 'ES11', 'ES12', 'ES13', 'ES21', 'ES22', 'ES23', 'ES24', 'ES30', 'ES41', 'ES42', 'ES43', 'ES51', 'ES52', 'ES53', 'ES61', 'ES62', 'ES63', 'ES64', 'ES7

In [ ]:
from datetime import datetime

df = eurostat.get_data_df(DEMOGRAPH)
dim = eurostat.get_pars(DEMOGRAPH)
demo_slice = df[(df["age"] == "TOTAL")
                & (df["sex"] == "T")
                & (df["geo\\TIME_PERIOD"].isin(geo_demo_nuts2))]
demo_vals = demo_slice[demo_slice.columns[len(dim):]]
print(f"Demographic's value range: {demo_vals.min().min()}, {demo_vals.max().max()}")

df_env = eurostat.get_data_df(ENVIRONMENT)
dim_env = eurostat.get_pars(ENVIRONMENT)
AGG = {"EU27_2020", "EU28", "EA19", "EA20"}
env_slice = df_env[(df_env["airpol"] == "GHG")
                   & (df_env["src_crf"] == "TOTX4_MEMO")
                   & (df_env["geo\\TIME_PERIOD"].isin(geo_env_country))]
env_vals = env_slice[env_slice.columns[len(dim_env):]]
print(f"Environmental emission value range: {env_vals.min().min()}, {env_vals.max().max()}")

print(f"Demographic's data staleness: {datetime.now().year - int(demo_slice.columns[len(dim):].max()) + 1} year(s)")
print(f"Environmental emission data staleness: {datetime.now().year - int(env_slice.columns[len(dim_env):].max()) + 1} year(s)")

Demographic's value range: 0.0, 15907951.0
Environmental emission value range: 1.83974, 1253127.92
Demographic's data staleness: 2 year(s)
Environmental emission data staleness: 3 year(s)


In [7]:
from eurostat_dq.ingest import fetch_dataset
from eurostat_dq.config import DATASETS

df = fetch_dataset("demo_r_d2jan")
df = fetch_dataset("demo_r_d2jan", use_cache=False)
df.shape
df.head()

Code found in cache
DataFrame acquired from the internet
DataFrame saved to cache


,freq,unit,sex,age,geo\TIME_PERIOD,1990,1991,1992,1993,1994,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,A,NR,F,TOTAL,AL,NaN,NaN,NaN,NaN,NaN,...,1417141.0,1423050.0,1431715.0,1432833.0,1425342.0,1419759.0,1406532.0,1394864.0,NaN,1194597.0
1,A,NR,F,TOTAL,AL0,NaN,NaN,NaN,NaN,NaN,...,1417141.0,1423050.0,1431715.0,1432833.0,1425342.0,1419759.0,1406532.0,1394864.0,NaN,1194597.0
2,A,NR,F,TOTAL,AL01,NaN,NaN,NaN,NaN,NaN,...,406682.0,405835.0,405598.0,404201.0,399599.0,396799.0,390886.0,385462.0,NaN,317141.0
3,A,NR,F,TOTAL,AL02,NaN,NaN,NaN,NaN,NaN,...,564402.0,574010.0,585530.0,590623.0,594008.0,596005.0,597622.0,598531.0,NaN,506383.0
4,A,NR,F,TOTAL,AL03,NaN,NaN,NaN,NaN,NaN,...,446057.0,443205.0,440587.0,438009.0,431735.0,426955.0,418024.0,410871.0,NaN,371073.0


In [ ]:
BASE = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"
resp = requests.get(f"{BASE}/demo_r_d2jan",
    params={"format":"JSON","lang":"en","age":"TOTAL","sex":"T"},
    timeout=30)
resp.raise_for_status()
d = resp.json()
print(json.dumps(d, indent=2))
# print(d["value"])
# print(d["dimension"]["time"]["category"]["index"])

In [13]:
df = eurostat.get_data_df(DEMOGRAPH)
dim = eurostat.get_pars(DEMOGRAPH)
demo_slice = df[(df["age"] == "TOTAL")
                & (df["sex"] == "T")
                & (df["geo\\TIME_PERIOD"] == "HU11")
                & (df["geo\\TIME_PERIOD"].isin(geo_demo_nuts2))]
demo_vals = demo_slice[demo_slice.columns[len(dim)+11:]]

demo_vals

,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
107442,1759209.0,1739569.0,1719342.0,1705309.0,1697343.0,1698106.0,1696128.0,1702297.0,1712210.0,1721556.0,...,1738570.0,1728929.0,1723033.0,1722363.0,1717144.0,1690503.0,1672443.0,1671004.0,1686222.0,1685209.0


In [ ]:
rows = [{"2001": "Ford", "2002": "Honda"}, {"2001": "Ford", "2002": "Honda"}]
data = pd.DataFrame(rows)

# keys = np.array([36])
# np.unravel_index(keys, [1,1,1,1,521,36])
# # -> (array([0,0,0]), ..., array([0,1,468]), array([0,0,0]))
# print(36//521)
# print(521//36)
# print(36%521)
# print(521%36)

resp = requests.get(f"https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/demo_r_d2jan",
    params={"format":"JSON", "lang":"en"},
    timeout=30)
resp.raise_for_status()
d = resp.json()
print(json.dumps(d, indent=2))

keys = np.array([int(k) for k in d["value"].keys()])
print(keys)
inds = np.unravel_index(keys, d["size"])
print(inds)

freq_inv = {int(v): k for k, v in d["dimension"]["freq"]["category"]["index"].items()}
unit_inv = {int(v): k for k, v in d["dimension"]["unit"]["category"]["index"].items()}
sex_inv = {int(v): k for k, v in d["dimension"]["sex"]["category"]["index"].items()}
age_inv = {int(v): k for k, v in d["dimension"]["age"]["category"]["index"].items()}
geo_inv = {int(v): k for k, v in d["dimension"]["geo"]["category"]["index"].items()}
time_inv = {int(v): k for k, v in d["dimension"]["time"]["category"]["index"].items()}

In [6]:
i = 20
for freq_ind, unit_ind, sex_ind, age_ind, geo_ind, time_ind, value in zip(inds[0], inds[1], inds[2], inds[3], inds[4], inds[5], d["value"].values()):
    print(f"FREQ: {freq_inv[freq_ind]}, UNIT: {unit_inv[unit_ind]}, SEX: {sex_inv[sex_ind]}, AGE: {age_inv[age_ind]}, GEO: {geo_inv[geo_ind]}, TIME: {time_inv[time_ind]}, VALUE: {value}")
    i -= 1
    if i == 0:
        break

FREQ: A, UNIT: NR, SEX: F, AGE: TOTAL, GEO: AL, TIME: 2000, VALUE: 1526762
FREQ: A, UNIT: NR, SEX: F, AGE: TOTAL, GEO: AL, TIME: 2001, VALUE: 1535822
FREQ: A, UNIT: NR, SEX: F, AGE: TOTAL, GEO: AL, TIME: 2002, VALUE: 1532563
FREQ: A, UNIT: NR, SEX: F, AGE: TOTAL, GEO: AL, TIME: 2003, VALUE: 1526180
FREQ: A, UNIT: NR, SEX: F, AGE: TOTAL, GEO: AL, TIME: 2004, VALUE: 1520481
FREQ: A, UNIT: NR, SEX: F, AGE: TOTAL, GEO: AL, TIME: 2005, VALUE: 1512745
FREQ: A, UNIT: NR, SEX: F, AGE: TOTAL, GEO: AL, TIME: 2006, VALUE: 1503969
FREQ: A, UNIT: NR, SEX: F, AGE: TOTAL, GEO: AL, TIME: 2007, VALUE: 1492439
FREQ: A, UNIT: NR, SEX: F, AGE: TOTAL, GEO: AL, TIME: 2008, VALUE: 1480358
FREQ: A, UNIT: NR, SEX: F, AGE: TOTAL, GEO: AL, TIME: 2009, VALUE: 1468935
FREQ: A, UNIT: NR, SEX: F, AGE: TOTAL, GEO: AL, TIME: 2010, VALUE: 1459025
FREQ: A, UNIT: NR, SEX: F, AGE: TOTAL, GEO: AL, TIME: 2011, VALUE: 1451691
FREQ: A, UNIT: NR, SEX: F, AGE: TOTAL, GEO: AL, TIME: 2012, VALUE: 1444234
FREQ: A, UNIT: NR, SEX: F